In [1]:
import os
import PyPDF2
import numpy as np
import pandas as pd
from tabula.io import read_pdf
from datetime import datetime
import re

In [2]:
file_name = r"C:\Users\admin\Downloads\22.11.2023 £2,996.86 Liqui Moly.pdf"

r"C:\Users\admin\Downloads\22.11.2023 £2,996.86 Liqui Moly.pdf"

'C:\\Users\\admin\\Downloads\\22.11.2023 £2,996.86 Liqui Moly.pdf'

In [3]:
invoice_type = "Products"

input_file = fr"C:\Users\admin\Downloads\02.07.2024 £754.94 Liqui Moly.pdf"

In [ ]:
name = "Liqui Moly"

table1 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(203,13,310,256),
                  columns=[132,256],
                  pandas_options={'header': None},
                  encoding="windows-1254")

heading = table1[0]
display(heading)

row_docnum = heading.index[heading[0] == "Invoice No."].tolist()[0]
docnum = heading[1][row_docnum]
print(docnum)

row_date = heading.index[heading[0] == "Posting Date"].tolist()[0]
date = heading[1][row_date]
date = str(datetime.strptime(date, "%d %B %Y"))
print(date)

row_ordernum = heading.index[heading[0] == "Order No."].tolist()[0]
ordernum = heading[1][row_ordernum]

transfernum = None
print(ordernum)
print(transfernum)


,0,1
0,Invoice No.,112826
1,VAT Registration No.,GB262814642
2,Your ref.,PO38786 - CREDIT LIMIT EXCEED
3,Your order no.,A171718
4,Order No.,111842
5,Posting Date,02 July 2024
6,Due Date,30 August 2024
7,Payment Terms,Current Month 30 Days


112826
2024-07-02 00:00:00
111842
None


In [5]:
with open(input_file,'rb') as pdf_file:
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    num_pages = len(pdf_reader.pages)

print(num_pages)

1


In [6]:
#create loop to go through the pages
all_content = []
for page in range(1, num_pages + 1):
    table2 = read_pdf(input_file,
                    pages=page,
                    silent=True,
                    guess=False,
                    area=(328,27,770,575),
                    columns=[84,310,348,393,448,511,575],
                    pandas_options={'header': None},
                    encoding="windows-1254")
    contenti = table2[0]
    all_content.append(contenti)

content = pd.concat(all_content).reset_index(drop=True)
display(content)

,0,1,2,3,4,5,6
0,1080,LUBRICANT FIX 50G,12,Piece,1.81,NaN,21.72
1,3381,PRO-LINE INJECTOR AND GLOW PLUG GREASE 20G,12,Piece,2.53,NaN,30.36
2,3379,PRO-LINE INJECTOR & GLOW PLUG DISMANTLING AID,6,Piece,5.39,NaN,32.34
3,NaN,400ML,NaN,NaN,NaN,NaN,NaN
4,25065,MARINE SINGLE GRADE SAE30 1LTR,6,Piece,4.49,NaN,26.94
5,21605,TOP TEC 4210 0W-30 5LTR,4,Piece,35.73,NaN,142.92
6,2520,DIESEL PURGE 1LTR,6,Piece,7.43,NaN,44.58
7,7988,DRUM FILL LEVEL INDICATOR 1 PCE,2,Piece,32.28,NaN,64.56
8,5111,Pro-Line Throttle Valve Cleaner 400ml,6,Piece,6.92,NaN,41.52
9,7389,PRO-LINE SILICONE SPRAY 400ML,6,Piece,3.81,NaN,22.86


In [7]:
content[0] = pd.to_numeric(content[0], errors='coerce')  # Remove anything that not number in column 0
content = content.dropna(subset=[0,2,4]).reset_index(drop=True) # Remove rows with NaN in column 0,2,4
#display(content)
content[[0]] = content[[0]].astype('int').astype('string') #remove float data type in SKU 

content

,0,1,2,3,4,5,6
0,1080,LUBRICANT FIX 50G,12,Piece,1.81,NaN,21.72
1,3381,PRO-LINE INJECTOR AND GLOW PLUG GREASE 20G,12,Piece,2.53,NaN,30.36
2,3379,PRO-LINE INJECTOR & GLOW PLUG DISMANTLING AID,6,Piece,5.39,NaN,32.34
3,25065,MARINE SINGLE GRADE SAE30 1LTR,6,Piece,4.49,NaN,26.94
4,21605,TOP TEC 4210 0W-30 5LTR,4,Piece,35.73,NaN,142.92
5,2520,DIESEL PURGE 1LTR,6,Piece,7.43,NaN,44.58
6,7988,DRUM FILL LEVEL INDICATOR 1 PCE,2,Piece,32.28,NaN,64.56
7,5111,Pro-Line Throttle Valve Cleaner 400ml,6,Piece,6.92,NaN,41.52
8,7389,PRO-LINE SILICONE SPRAY 400ML,6,Piece,3.81,NaN,22.86
9,1606,MOTORBIKE FORK OIL 10W MEDIUM 5LTR,4,Piece,25.70,NaN,102.80


In [8]:
content.rename(columns={
    0: 'Item',
    1: 'Description',
    2: 'Qty',
    3: 'Unit',
    4: 'Unit Price',
    5: 'Discount %',
    6: 'Amount'}, inplace=True)

display(content)

,Item,Description,Qty,Unit,Unit Price,Discount %,Amount
0,1080,LUBRICANT FIX 50G,12,Piece,1.81,NaN,21.72
1,3381,PRO-LINE INJECTOR AND GLOW PLUG GREASE 20G,12,Piece,2.53,NaN,30.36
2,3379,PRO-LINE INJECTOR & GLOW PLUG DISMANTLING AID,6,Piece,5.39,NaN,32.34
3,25065,MARINE SINGLE GRADE SAE30 1LTR,6,Piece,4.49,NaN,26.94
4,21605,TOP TEC 4210 0W-30 5LTR,4,Piece,35.73,NaN,142.92
5,2520,DIESEL PURGE 1LTR,6,Piece,7.43,NaN,44.58
6,7988,DRUM FILL LEVEL INDICATOR 1 PCE,2,Piece,32.28,NaN,64.56
7,5111,Pro-Line Throttle Valve Cleaner 400ml,6,Piece,6.92,NaN,41.52
8,7389,PRO-LINE SILICONE SPRAY 400ML,6,Piece,3.81,NaN,22.86
9,1606,MOTORBIKE FORK OIL 10W MEDIUM 5LTR,4,Piece,25.70,NaN,102.80


In [9]:
dict_content = content.to_dict(orient='records')
dict_content

line_items=[]
for item in dict_content:

    partNum = item['Item']
    desc = item['Description']
    quantity = item['Qty']
    netTotal = item['Amount']

    print(partNum)
    
    line_item = {"line_type": "inventory",
                        "sku": str(partNum),
                        "name": desc,
                        "quantity": int(quantity),
                        "net_total": float(netTotal),
                        "tax_type": "INPUT2"}
    
    line_items.append(line_item)
    
print(line_items)

1080
3381
3379
25065
21605
2520
7988
5111
7389
1606
20754
[{'line_type': 'inventory', 'sku': '1080', 'name': 'LUBRICANT FIX 50G', 'quantity': 12, 'net_total': 21.72, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': '3381', 'name': 'PRO-LINE INJECTOR AND GLOW PLUG GREASE 20G', 'quantity': 12, 'net_total': 30.36, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': '3379', 'name': 'PRO-LINE INJECTOR & GLOW PLUG DISMANTLING AID', 'quantity': 6, 'net_total': 32.34, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': '25065', 'name': 'MARINE SINGLE GRADE SAE30 1LTR', 'quantity': 6, 'net_total': 26.94, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': '21605', 'name': 'TOP TEC 4210 0W-30 5LTR', 'quantity': 4, 'net_total': 142.92, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': '2520', 'name': 'DIESEL PURGE 1LTR', 'quantity': 6, 'net_total': 44.58, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': '7988', 'name': 'DRUM FILL LEVEL INDICATOR 1 PCE', 'quan

In [10]:
#create loop to go through the pages
all_total_content = []

for page in range(1, num_pages + 1):
    table3 = read_pdf(input_file,
                    pages=page,
                    silent=True,
                    guess=False,
                    area=(310,395,770,575),
                    columns=[500,575],
                    pandas_options={'header': None},
                    encoding='windows-1254')
    total_contenti=table3[0]
    all_total_content.append(total_contenti)

total_content = pd.concat(all_total_content).reset_index(drop=True)
display(total_content)



,0,1
0,Unit Price Discount %,Amount
1,1.81,21.72
2,2.53,30.36
3,5.39,32.34
4,4.49,26.94
5,35.73,142.92
6,7.43,44.58
7,32.28,64.56
8,6.92,41.52
9,3.81,22.86


In [11]:
row_index = total_content.index[total_content[0] == "Total GBP Incl. VAT"].tolist()[0]

final_total = float(str(total_content[1][row_index]).replace(',',''))
display(final_total)

754.94

In [12]:
payload = {}
keys = ["Source File",
        "Type",
        "Name",
        "Date",
        "Reference No.",
        "Order No.",
        "Transfer No.",
        "Document No.",
        "Line Items",
        "Total"]

values = [file_name,
        invoice_type,
        name,
        date,
        docnum,
        ordernum,
        transfernum,
        None,
        line_items,
        final_total]

for i, key in enumerate(keys):
    payload[key] = values[i]

payload

{'Source File': 'C:\\Users\\admin\\Downloads\\22.11.2023 £2,996.86 Liqui Moly.pdf',
 'Type': 'Products',
 'Name': 'Liqui Moly',
 'Date': '2024-07-02 00:00:00',
 'Reference No.': '112826',
 'Order No.': '111842',
 'Transfer No.': None,
 'Document No.': None,
 'Line Items': [{'line_type': 'inventory',
   'sku': '1080',
   'name': 'LUBRICANT FIX 50G',
   'quantity': 12,
   'net_total': 21.72,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': '3381',
   'name': 'PRO-LINE INJECTOR AND GLOW PLUG GREASE 20G',
   'quantity': 12,
   'net_total': 30.36,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': '3379',
   'name': 'PRO-LINE INJECTOR & GLOW PLUG DISMANTLING AID',
   'quantity': 6,
   'net_total': 32.34,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': '25065',
   'name': 'MARINE SINGLE GRADE SAE30 1LTR',
   'quantity': 6,
   'net_total': 26.94,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': '21605',
   'name': 'TOP TEC 4210 0W